<a href="https://colab.research.google.com/github/Samuelcrtlc/engenharia_de_prompt_ia/blob/main/Aula08_Miss%C3%A3o_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Missão 2: Consulta de Endereço via ViaCEP API (Comentado)

###Nome : Samuel Natalicio da Silva

Este script permite consultar informações de endereço utilizando a API pública do ViaCEP. Cada linha de código crucial está comentada para facilitar o entendimento.

In [ ]:
import requests # Importa a biblioteca 'requests' para fazer requisições HTTP (acesso à internet).
import json     # Importa a biblioteca 'json' para trabalhar com dados no formato JSON.

def consultar_cep(cep):
    """
    Consulta a API ViaCEP para obter informações de endereço de um determinado CEP.

    Args:
        cep (str): O CEP a ser consultado (apenas números).

    Returns:
        dict or None: Um dicionário com os dados do endereço se a consulta for bem-sucedida,
                      ou None em caso de erro.
    """
    # Remove qualquer caractere não numérico do CEP (e.g., '-', '.') para garantir que tenhamos apenas dígitos.
    cep = ''.join(filter(str.isdigit, cep))

    # Verifica se o CEP resultante tem 8 dígitos. Se não tiver, é considerado inválido.
    if len(cep) != 8:
        print(f"Erro: O CEP '{cep}' é inválido. Um CEP deve conter 8 dígitos numéricos.")
        return None # Retorna None indicando falha na consulta.

    # Constrói a URL para a requisição à API do ViaCEP usando o CEP fornecido.
    url = f"https://viacep.com.br/ws/{cep}/json/"

    # Bloco try-except para lidar com possíveis erros durante a requisição à API.
    try:
        # Faz uma requisição GET para a URL do ViaCEP com um tempo limite de 5 segundos.
        response = requests.get(url, timeout=5)
        # Verifica se a requisição foi bem-sucedida (status code 2xx). Se não, levanta um erro HTTP.
        response.raise_for_status()

        # Converte a resposta JSON da API em um dicionário Python.
        data = response.json()

        # Verifica se a resposta da API contém uma chave 'erro' com valor True, indicando que o CEP não foi encontrado.
        if 'erro' in data and data['erro']:
            print(f"Erro: CEP '{cep}' não encontrado ou inválido pela API do ViaCEP.")
            return None # Retorna None indicando que o CEP não existe ou é inválido.

        return data # Retorna o dicionário com os dados do endereço.

    except requests.exceptions.Timeout:
        # Captura o erro se a requisição exceder o tempo limite.
        print(f"Erro de Conexão: A requisição excedeu o tempo limite ao tentar acessar o ViaCEP para o CEP '{cep}'. Verifique sua conexão.")
        return None
    except requests.exceptions.ConnectionError:
        # Captura o erro se não houver conexão com a internet ou o servidor estiver inacessível.
        print(f"Erro de Conexão: Não foi possível conectar ao servidor do ViaCEP para o CEP '{cep}'. Verifique sua conexão com a internet.")
        return None
    except requests.exceptions.HTTPError as e:
        # Captura erros HTTP (4xx ou 5xx) que não foram tratados pelo 'erro' da API.
        print(f"Erro HTTP ao consultar o ViaCEP para o CEP '{cep}': {e}")
        return None
    except json.JSONDecodeError:
        # Captura o erro se a resposta da API não for um JSON válido.
        print(f"Erro: Resposta inválida da API do ViaCEP para o CEP '{cep}'. Não foi possível decodificar o JSON.")
        return None
    except Exception as e:
        # Captura qualquer outro erro inesperado que possa ocorrer.
        print(f"Ocorreu um erro inesperado ao consultar o CEP '{cep}': {e}")
        return None

def imprimir_endereco(endereco_data):
    """
    Formata e imprime as informações de endereço de forma legível.

    Args:
        endereco_data (dict): Dicionário contendo os dados do endereço.
    """
    # Verifica se há dados de endereço para imprimir (ou seja, se a consulta foi bem-sucedida).
    if endereco_data:
        print("\n--- Informações do Endereço ---") # Cabeçalho para a saída formatada.
        # Usa .get() para acessar os valores do dicionário, fornecendo 'N/A' se a chave não existir.
        print(f"CEP: {endereco_data.get('cep', 'N/A')}")
        print(f"Logradouro: {endereco_data.get('logradouro', 'N/A')}")
        print(f"Complemento: {endereco_data.get('complemento', 'N/A')}")
        print(f"Bairro: {endereco_data.get('bairro', 'N/A')}")
        print(f"Localidade: {endereco_data.get('localidade', 'N/A')}")
        print(f"UF: {endereco_data.get('uf', 'N/A')}")
        print(f"DDD: {endereco_data.get('ddd', 'N/A')}")
        print("--------------------------------") # Rodapé da saída formatada.
    else:
        print("Não há informações de endereço para exibir.") # Mensagem se não houver dados válidos.

### Exemplos de Uso (Comentado):

Vamos testar as funções com diferentes CEPs para demonstrar seu funcionamento, incluindo tratamento de erros.

In [ ]:
# --- Exemplo de Uso ---

# Exemplo 1: Consulta com um CEP válido e formatação da saída.
print("\n--- Teste com CEP válido (01001-000) ---")
cep_valido = "01001-000"
endereco_valido = consultar_cep(cep_valido) # Chama a função para consultar o CEP.
imprimir_endereco(endereco_valido)       # Chama a função para imprimir o resultado.

# Exemplo 2: Consulta com outro CEP válido.
print("\n--- Teste com outro CEP válido (70000-000) ---")
cep_outro_valido = "70000-000"
endereco_outro_valido = consultar_cep(cep_outro_valido)
imprimir_endereco(endereco_outro_valido)

# Exemplo 3: Consulta com um CEP que não existe (a API ViaCEP retornará 'erro: true').
print("\n--- Teste com CEP inválido (99999-999 - não existente) ---")
cep_nao_existente = "99999-999"
endereco_nao_existente = consultar_cep(cep_nao_existente)
imprimir_endereco(endereco_nao_existente) # Imprimirá 'Não há informações de endereço para exibir.'

# Exemplo 4: Consulta com CEP com formato incorreto (menos dígitos). A função 'consultar_cep' tratará isso.
print("\n--- Teste com CEP com formato incorreto (12345-67 - menos dígitos) ---")
cep_formato_incorreto = "12345-67"
endereco_formato_incorreto = consultar_cep(cep_formato_incorreto)
imprimir_endereco(endereco_formato_incorreto)

# Exemplo 5: Consulta com CEP com formato incorreto (mais dígitos).
print("\n--- Teste com CEP com formato incorreto (12345-6789 - mais dígitos) ---")
cep_formato_incorreto_longo = "12345-6789"
endereco_formato_incorreto_longo = consultar_cep(cep_formato_incorreto_longo)
imprimir_endereco(endereco_formato_incorreto_longo)

# Exemplo 6: Consulta com CEP contendo letras. A função 'consultar_cep' filtrará as letras.
print("\n--- Teste com CEP contendo letras (A01B001-000C) ---")
cep_com_letras = "A01B001-000C"
endereco_com_letras = consultar_cep(cep_com_letras)
imprimir_endereco(endereco_com_letras)

### Consultar CEP Personalizado (Comentado)

Use o campo abaixo para inserir um CEP e consultá-lo interativamente.

In [ ]:
# Solicita que o usuário digite um CEP para consulta.
meu_cep = input("Digite o CEP que deseja consultar (apenas números ou com hífen): ")

# Chama a função 'consultar_cep' com o CEP fornecido pelo usuário.
endereco_personalizado = consultar_cep(meu_cep)

# Chama a função 'imprimir_endereco' para exibir os dados do endereço de forma formatada.
imprimir_endereco(endereco_personalizado)